# Five-day exact-Hessian compile benchmark

This standalone A100 benchmark isolates one question: does `torch.compile` become worthwhile for the original five-day exact-Hessian collocation problem?

It compares two CUDA arms using the corrected, vmap-safe scaling-and-squaring matrix exponential:

- **eager_ss_fixed**: eager exact Hessian.
- **compiled_ss_fixed**: `torch.compile(fullgraph=True)` exact Hessian.

Both arms use the same five-day workload and stop after 20 IPOPT iterations. The first compiled Hessian includes compilation; warmed timings measure steady-state performance. This is a dispatch benchmark, not a convergence comparison. Native compilation is excluded because PyTorch's generated higher-order `matrix_exp` graph fails inside Inductor on CUDA.

The final cell also compares against the previously measured 24-hour timings to show scaling of cold compilation and warmed Hessian execution.

In [ ]:
import json
import os
from pathlib import Path
import platform
import subprocess
import sys

import torch

REPO_URL = "https://github.com/JBjoernskov/Twin4Build.git"
CANDIDATE_REF = "feature/issue-126/reduce-hessian-dispatch"
ROOT = Path("/content/twin4build_full_compile_hessian")
CHECKOUT = ROOT / "candidate"

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA Colab runtime is required.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", f"git+{REPO_URL}@{CANDIDATE_REF}"],
    check=True,
)
ROOT.mkdir(parents=True, exist_ok=True)
if CHECKOUT.exists():
    subprocess.run(["git", "fetch", "--quiet", "origin", CANDIDATE_REF], cwd=CHECKOUT, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{CANDIDATE_REF}"], cwd=CHECKOUT, check=True)
else:
    subprocess.run(
        ["git", "clone", "--quiet", "--depth", "1", "--branch", CANDIDATE_REF, REPO_URL, str(CHECKOUT)],
        check=True,
    )

ss_source = (CHECKOUT / "twin4build/systems/utils/discrete_statespace_system.py").read_text()
required = ["delta = 2.0 * delta + delta @ delta", "compile_hessian"]
transcription_source = (CHECKOUT / "twin4build/estimator/_transcription.py").read_text()
if required[0] not in ss_source or required[1] not in transcription_source:
    raise RuntimeError("Candidate checkout lacks the corrected SS or compiled Hessian path.")

props = torch.cuda.get_device_properties(0)
print(json.dumps({
    "gpu": props.name,
    "gpu_memory_gb": props.total_memory / 1e9,
    "torch": torch.__version__,
    "python": platform.python_version(),
    "candidate_ref": CANDIDATE_REF,
}, indent=2))

In [ ]:
# Reuse the production-workload runner maintained by the focused dispatch
# benchmark, while keeping this notebook's results in an independent directory.
dispatch_notebook = json.loads(
    (CHECKOUT / "twin4build/examples/gpu_hessian_dispatch_benchmark.ipynb").read_text()
)
runner_cells = [
    "".join(cell.get("source", []))
    for cell in dispatch_notebook["cells"]
    if cell.get("cell_type") == "code"
    and "RUNNER = ROOT / \"run_hessian_arm.py\"" in "".join(cell.get("source", []))
]
if len(runner_cells) != 1:
    raise RuntimeError("Could not identify the exact-Hessian benchmark runner cell.")
exec(compile(runner_cells[0], "gpu_hessian_dispatch runner", "exec"), globals())

In [ ]:
BENCH_HOURS = 120
BENCH_MAXITER = 20
CURRENT_REF = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=CHECKOUT, text=True
).strip()
RESULT_FILES = {
    "eager_ss": ROOT / "eager_ss_fixed_120h.json",
    "compiled_ss": ROOT / "compiled_ss_fixed_120h.json",
}


def reusable(path, label):
    if not path.exists():
        return False
    try:
        row = json.loads(path.read_text())
        return (
            row.get("ref") == CURRENT_REF
            and row.get("arm") == label
            and row.get("hours") == BENCH_HOURS
            and row.get("maxiter") == BENCH_MAXITER
        )
    except (json.JSONDecodeError, OSError):
        return False


for label, result_file in RESULT_FILES.items():
    if reusable(result_file, label):
        print(f"Reusing current {label} result.")
        continue
    print(f"\nRunning five-day {label} exact Hessian ...", flush=True)
    env = os.environ.copy()
    env["T4B_BENCH_HOURS"] = str(BENCH_HOURS)
    env["T4B_BENCH_MAXITER"] = str(BENCH_MAXITER)
    env["TWIN4BUILD_TRANSFORM_MATRIX_EXP"] = "ss"
    env["PYTHONPATH"] = str(CHECKOUT) + os.pathsep + env.get("PYTHONPATH", "")
    completed = subprocess.run(
        [sys.executable, str(RUNNER), str(CHECKOUT), label, str(result_file)],
        cwd=CHECKOUT,
        env=env,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout, flush=True)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr, flush=True)
    if completed.returncode:
        raise RuntimeError(
            f"{label} failed with exit code {completed.returncode}.\n"
            + "\n".join(completed.stderr.splitlines()[-80:])
        )

rows = [json.loads(path.read_text()) for path in RESULT_FILES.values()]
print("\nBoth five-day compile arms completed.")

In [ ]:
import math
import pandas as pd

REFERENCE_24H = {
    "eager_first_seconds": 0.517562453,
    "eager_warmed_median_seconds": 0.392828581,
    "compiled_first_seconds": 239.165338908,
    "compiled_warmed_median_seconds": 0.051575127,
}

summary = pd.DataFrame(rows)
eager = summary.loc[summary.arm == "eager_ss"].iloc[0]
compiled = summary.loc[summary.arm == "compiled_ss"].iloc[0]
summary["solver_speedup_vs_eager"] = eager.solver_seconds / summary.solver_seconds
summary["hessian_total_speedup_vs_eager"] = (
    eager.hessian_total_seconds / summary.hessian_total_seconds
)
summary["hessian_warmed_speedup_vs_eager"] = (
    eager.hessian_warmed_median_seconds / summary.hessian_warmed_median_seconds
)

compile_overhead = max(
    0.0, compiled.hessian_first_seconds - compiled.hessian_warmed_median_seconds
)
warmed_saving = (
    eager.hessian_warmed_median_seconds - compiled.hessian_warmed_median_seconds
)
break_even_calls = (
    math.inf if warmed_saving <= 0 else 1.0 + compile_overhead / warmed_saving
)

pd.set_option("display.max_columns", None)
display(summary)

print(
    f"Five-day warmed Hessian speedup: "
    f"{eager.hessian_warmed_median_seconds / compiled.hessian_warmed_median_seconds:.3f}x\n"
    f"Five-day total Hessian speedup (includes compile): "
    f"{eager.hessian_total_seconds / compiled.hessian_total_seconds:.3f}x\n"
    f"Approximate compile overhead: {compile_overhead:.1f} s\n"
    f"Estimated break-even Hessian calls: {break_even_calls:.0f}\n\n"
    f"Scaling from 24 h to 120 h:\n"
    f"  eager warmed Hessian: "
    f"{eager.hessian_warmed_median_seconds / REFERENCE_24H['eager_warmed_median_seconds']:.3f}x\n"
    f"  compiled warmed Hessian: "
    f"{compiled.hessian_warmed_median_seconds / REFERENCE_24H['compiled_warmed_median_seconds']:.3f}x\n"
    f"  compiled first call: "
    f"{compiled.hessian_first_seconds / REFERENCE_24H['compiled_first_seconds']:.3f}x"
)

if eager.status != compiled.status:
    print("WARNING: solver statuses differ; inspect objective and defect before accepting timing.")
objective_scale = max(1.0, abs(eager.objective))
if abs(eager.objective - compiled.objective) > 1e-6 * objective_scale:
    print("WARNING: objectives differ materially; reject the compiled timing comparison.")
if abs(eager.max_defect - compiled.max_defect) > 1e-5:
    print("WARNING: continuity defects differ materially; reject the compiled timing comparison.")